# ARC-AGI-3 Memory Agent (v6) — cross-game shared memory

## What changed from v5

In v5, the framework instantiated a **new `MyAgent` object per game run**, so
`self.tmem` in `__init__` was always empty at the start of each game. The agent
re-learned the same mechanics from scratch every time.

v6 introduces a **class-level shared memory dict** that persists for the entire
process lifetime:

```python
class MyAgent(Agent):
    _SHARED_MEMORY: dict = {}  # class-level, survives across instances

    def __init__(self, ...):
        game_type = self.game_id.split('-')[0]  # 'ls20' from 'ls20-abc123'
        if game_type not in MyAgent._SHARED_MEMORY:
            MyAgent._SHARED_MEMORY[game_type] = TransitionMemory(4000, 4000)
        self.tmem = MyAgent._SHARED_MEMORY[game_type]  # attach, don't create
```

## Memory scope comparison

| Scope | v5 | v6 |
|-------|----|----|
| Same level | ✓ shared | ✓ shared |
| Next level | ✓ shared | ✓ shared |
| Next game instance (same type) | ✗ empty | **✓ shared** |
| Different game type | — | ✓ isolated |

## Why game_type isolation matters

The competition runs multiple game types (ls20, vc33, ft09, ...) in the same
process. Each game type has distinct mechanics, so mixing their memories would
corrupt the bias signal. Keying by the prefix before the first `-` in `game_id`
isolates them correctly. A `game_id` with no `-` falls back to the full id.

## Memory capacity raised: 1000 → 4000 per buffer

Since memory now accumulates across multiple game runs (not just levels),
the ring-buffer capacity is raised from 1000 to 4000 entries for both
frame memory and click memory. At ~1KB per entry this is ~4MB per game type — negligible.

## All v5 components unchanged

`extract_objects`, `object_descriptor`, `frame_descriptor_v2`, `click_heatmap`,
two-buffer `TransitionMemory`, and all v4 fixes (A, B, C) and v3 bug-fixes
(FIX 1–6) are retained exactly as in v5.


In [ ]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

In [ ]:
%%writefile /kaggle/working/my_agent.py
# =============================================================================
# ARC3-Memory Agent  (v6) — cross-game shared memory
#
# Core upgrade over v5:
#   TransitionMemory is now SHARED ACROSS ALL GAME INSTANCES of the same
#   game type via a class-level dict (_SHARED_MEMORY).
#
#   In v5, the framework instantiates a new MyAgent object for each game run.
#   Because self.tmem lived in __init__, every new game started with empty
#   memory — re-learning the same mechanics from scratch each time.
#
#   In v6, memories are keyed by game_type (the prefix before the first '-'
#   in game_id, e.g. "ls20" from "ls20-abc123"). All instances of the same
#   game share one TransitionMemory for the process lifetime. By game
#   instance 2, the agent already knows which object types caused frame
#   changes in game instance 1.
#
#   Implementation:
#     MyAgent._SHARED_MEMORY: dict[str, TransitionMemory]  (class-level)
#     game_type = self.game_id.split('-')[0]
#     self.tmem = _SHARED_MEMORY.setdefault(game_type, TransitionMemory(...))
#
#   Memory scope (v6):
#     Same level   → tmem shared ✓  (unchanged from v5)
#     Next level   → tmem shared ✓  (unchanged from v5)
#     Next game instance (same type) → tmem shared ✓  (NEW in v6)
#     Different game type            → separate tmem   (correct isolation)
#
#   Capacity is raised to 4000/4000 (from 1000/1000) since memory now
#   accumulates across multiple game runs.
#
# All v5 components retained: extract_objects, object_descriptor,
# frame_descriptor_v2, click_heatmap, two-buffer TransitionMemory,
# and all v4 fixes (A, B, C) and v3 bug-fixes (FIX 1-6).
# =============================================================================

import hashlib
import logging
import os
import random
import time
import traceback
from collections import deque
from typing import Any, List, Dict, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from agents.agent import Agent
from arcengine import FrameData, GameAction, GameState

logger = logging.getLogger(__name__)

_UNDO_ACTION_IDX = -1   # sentinel: prev_action was UNDO → skip memory store


# =============================================================================
# Object extraction (BFS connected components, pure numpy)
# =============================================================================

def extract_objects(
    frame: np.ndarray,
    bg: int,
    min_size: int = 4,
    max_size: int = 3000,
) -> List[Dict]:
    """
    Return a list of objects (connected pixel blobs) in the frame.

    Each object is a dict:
      color : int        — color index (0-15)
      cx    : float      — centroid x (column), 0-63
      cy    : float      — centroid y (row), 0-63
      size  : int        — pixel count
      w     : int        — bounding box width
      h     : int        — bounding box height

    Algorithm: per-color DFS flood-fill (stack-based).
    Performance: ~0.5ms/frame for typical ARC-AGI-3 frames.
    """
    objects = []
    visited_global = np.zeros((64, 64), dtype=bool)

    for color in range(16):
        if color == bg:
            continue
        color_mask = (frame == color)
        if not color_mask.any():
            continue

        # Build per-color coordinate set for O(1) neighbour lookup
        ys, xs = np.where(color_mask & ~visited_global)
        if len(ys) == 0:
            continue

        pos_set = set(zip(ys.tolist(), xs.tolist()))
        processed = set()

        for i in range(len(ys)):
            sy, sx = int(ys[i]), int(xs[i])
            if (sy, sx) in processed:
                continue

            # DFS — stack, not recursion (avoids Python recursion limit)
            stack = [(sy, sx)]
            blob_ys, blob_xs = [], []

            while stack:
                y, x = stack.pop()
                if (y, x) in processed:
                    continue
                processed.add((y, x))
                blob_ys.append(y)
                blob_xs.append(x)
                for ny, nx in ((y-1,x),(y+1,x),(y,x-1),(y,x+1)):
                    if (ny, nx) in pos_set and (ny, nx) not in processed:
                        stack.append((ny, nx))

            n = len(blob_ys)
            if min_size <= n <= max_size:
                visited_global[blob_ys, blob_xs] = True
                objects.append({
                    'color': color,
                    'cx':   float(sum(blob_xs)) / n,
                    'cy':   float(sum(blob_ys)) / n,
                    'size': n,
                    'w':    max(blob_xs) - min(blob_xs) + 1,
                    'h':    max(blob_ys) - min(blob_ys) + 1,
                })

    objects.sort(key=lambda o: -o['size'])
    return objects


def find_object_at(objects: List[Dict], y: int, x: int) -> Optional[Dict]:
    """
    Return the object whose centroid is closest to pixel (y, x).
    Accepts if distance ≤ sqrt(size) + 2 (generous for click registration).
    Returns None if no object is close enough.
    """
    best, best_dist = None, float('inf')
    for obj in objects:
        d = abs(obj['cy'] - y) + abs(obj['cx'] - x)
        if d < best_dist:
            best_dist = d
            best = obj
    if best is not None and best_dist <= (best['size'] ** 0.5 + 2):
        return best
    return None


# =============================================================================
# Descriptors
# =============================================================================

def object_descriptor(obj: Dict) -> np.ndarray:
    """
    21-dim float32 descriptor for a single object.
      [0:16]  one-hot color
      16      cx / 64      (normalised column centroid)
      17      cy / 64      (normalised row centroid)
      18      size / 4096  (normalised pixel count)
      19      w / 64       (normalised width)
      20      h / 64       (normalised height)

    Cosine similarity between two object descriptors is high when:
      - same color (dominant term, 16 dims)
      - similar position (secondary, 2 dims)
      - similar size / shape (tertiary, 3 dims)
    This means "yellow cube here" ≈ "yellow cube there" (high sim),
    but "yellow cube" ≠ "red cube" (low sim). Exactly what we want.
    """
    d = np.zeros(21, dtype=np.float32)
    d[obj['color']] = 1.0
    d[16] = obj['cx'] / 64.0
    d[17] = obj['cy'] / 64.0
    d[18] = obj['size'] / 4096.0
    d[19] = obj['w'] / 64.0
    d[20] = obj['h'] / 64.0
    return d


def frame_descriptor_v2(objects: List[Dict], bg: int) -> np.ndarray:
    """
    80-dim object-centric frame descriptor.

    For each of 16 colors (5 features, base = color * 5):
      base+0  n_objects of this color, capped at 10, normalised
      base+1  total pixels of this color / 4096
      base+2  cx of largest object / 64    (0 if color absent)
      base+3  cy of largest object / 64    (0 if color absent)
      base+4  size of largest object / 4096 (0 if color absent)

    Why this is better than the 48-dim histogram:
      - Encodes WHERE objects are (centroid), not just HOW MANY pixels.
      - "One red blob in top-left" vs "one red blob in bottom-right" have
        different descriptors, even though histogram is identical.
      - Memory retrieval is now layout-sensitive: states from different levels
        where objects occupy different positions produce different similarity scores.
    """
    desc = np.zeros(16 * 5, dtype=np.float32)
    by_color: Dict[int, List] = {}
    for obj in objects:
        c = obj['color']
        if c == bg:
            continue
        if c not in by_color:
            by_color[c] = []
        by_color[c].append(obj)

    for c, objs in by_color.items():
        base = c * 5
        largest = max(objs, key=lambda o: o['size'])
        desc[base + 0] = min(len(objs), 10) / 10.0
        desc[base + 1] = sum(o['size'] for o in objs) / 4096.0
        desc[base + 2] = largest['cx'] / 64.0
        desc[base + 3] = largest['cy'] / 64.0
        desc[base + 4] = largest['size'] / 4096.0

    return desc


# =============================================================================
# TransitionMemory (v2) — split frame memory + click object memory
# =============================================================================

class TransitionMemory:
    """
    Two ring-buffers with object-centric representations:

    _frame_mem  — directional action transitions
        key:    80-dim frame_descriptor_v2
        action: 0-4 (ACTION1-5)
        reward: float (only frame-changing transitions stored)

    _click_mem  — click action transitions
        key:    21-dim object_descriptor of the clicked object
        reward: float (only frame-changing transitions stored)

    Persists ACROSS levels. UNDO transitions not stored. No-op transitions
    not stored (v4 FIX A).

    Query interface:
        discrete_bias(frame_desc) → np.ndarray (5,)
            Bias for ACTION1-5 logits based on similar past directional transitions.

        click_heatmap(objects)    → np.ndarray (64, 64)
            Per-pixel click score: Gaussian bumps at centroids of objects
            that are similar to previously-rewarding clicked objects.
            Shape (64,64) can be flattened to (4096,) and added to coord logits.
    """

    def __init__(self, frame_maxsize: int = 1000, click_maxsize: int = 1000):
        # Frame memory (directional)
        self._f_descs   = np.zeros((frame_maxsize, 80), dtype=np.float32)
        self._f_actions = np.zeros(frame_maxsize, dtype=np.int32)
        self._f_rewards = np.zeros(frame_maxsize, dtype=np.float32)
        self._f_ptr, self._f_size = 0, 0
        self._f_max = frame_maxsize

        # Click object memory
        self._c_descs   = np.zeros((click_maxsize, 21), dtype=np.float32)
        self._c_rewards = np.zeros(click_maxsize, dtype=np.float32)
        self._c_ptr, self._c_size = 0, 0
        self._c_max = click_maxsize

    def store_directional(
        self, frame_desc: np.ndarray, action_class: int, reward: float
    ) -> None:
        """Store a directional transition (action_class 0-4)."""
        self._f_descs[self._f_ptr]   = frame_desc
        self._f_actions[self._f_ptr] = action_class
        self._f_rewards[self._f_ptr] = reward
        self._f_ptr  = (self._f_ptr + 1) % self._f_max
        self._f_size = min(self._f_size + 1, self._f_max)

    def store_click(self, obj_desc: np.ndarray, reward: float) -> None:
        """Store a click transition indexed by the clicked object's descriptor."""
        self._c_descs[self._c_ptr]   = obj_desc
        self._c_rewards[self._c_ptr] = reward
        self._c_ptr  = (self._c_ptr + 1) % self._c_max
        self._c_size = min(self._c_size + 1, self._c_max)

    def discrete_bias(
        self, frame_desc: np.ndarray, k: int = 8, strength: float = 0.8
    ) -> np.ndarray:
        """
        5-dim bias vector for directional action logits.
        Positive = historically rewarding; negative = historically penalising.
        """
        bias = np.zeros(5, dtype=np.float32)
        if self._f_size == 0:
            return bias

        q = frame_desc
        qn = np.linalg.norm(q)
        if qn < 1e-8:
            return bias

        valid = self._f_descs[:self._f_size]
        sims  = valid.dot(q) / (np.linalg.norm(valid, axis=1).clip(1e-8) * qn)
        k_act = min(k, self._f_size)
        top   = np.argpartition(sims, -k_act)[-k_act:]

        total = sims[top].sum() + 1e-8
        for i in top:
            act = int(self._f_actions[i])
            if 0 <= act <= 4:
                bias[act] += (sims[i] / total) * self._f_rewards[i]

        return np.clip(bias, -3.0, 3.0) * strength

    def click_heatmap(
        self,
        objects: List[Dict],
        k: int = 8,
        sigma: float = 4.0,
        strength: float = 1.0,
    ) -> np.ndarray:
        """
        (64, 64) click score map.

        For each current object:
          1. Compute its similarity to all stored clicked objects.
          2. Compute a score = sum(sim × reward) over k nearest.
          3. Add a Gaussian bump of height=score at the object's centroid.

        The bump shape makes neighbouring pixels of a rewarding object
        also receive elevated click probability, which is correct since
        click registration is spatially fuzzy in most ARC-AGI-3 games.

        Returns zeros if click memory is empty or no objects present.
        """
        heatmap = np.zeros((64, 64), dtype=np.float32)
        if self._c_size == 0 or not objects:
            return heatmap

        mem_arr = self._c_descs[:self._c_size]    # (M, 21)
        mem_rew = self._c_rewards[:self._c_size]  # (M,)
        mem_nrm = np.linalg.norm(mem_arr, axis=1).clip(1e-8)

        ys_g = np.arange(64, dtype=np.float32)[:, None]  # (64,1)
        xs_g = np.arange(64, dtype=np.float32)[None, :]  # (1,64)
        two_sig2 = 2.0 * sigma ** 2

        for obj in objects:
            od  = object_descriptor(obj)
            odn = np.linalg.norm(od)
            if odn < 1e-8:
                continue

            sims = mem_arr.dot(od) / (mem_nrm * odn)  # (M,)
            k_act = min(k, self._c_size)
            top   = np.argpartition(sims, -k_act)[-k_act:]
            total = sims[top].sum() + 1e-8
            score = float(np.sum((sims[top] / total) * mem_rew[top])) * strength

            if score <= 0:
                continue  # don't suppress — just skip negative-score objects
                           # (they're already penalised by coord masking in CNN)

            cy, cx = obj['cy'], obj['cx']
            bump = score * np.exp(-((ys_g - cy)**2 + (xs_g - cx)**2) / two_sig2)
            heatmap += bump

        return heatmap

    @property
    def frame_size(self): return self._f_size

    @property
    def click_size(self): return self._c_size


# =============================================================================
# CNN architecture  (unchanged from v4)
# =============================================================================

class BaselineNet(nn.Module):
    def __init__(self, in_channels: int = 21, grid: int = 64):
        super().__init__()
        self.grid = grid
        self.conv1 = nn.Conv2d(in_channels,  32, 3, padding=1)
        self.conv2 = nn.Conv2d(32,  64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.conv4 = nn.Conv2d(128, 256, 3, padding=1)
        self.dropout = nn.Dropout(0.15)
        self.act_pool = nn.AdaptiveAvgPool2d(8)
        self.act_fc1  = nn.Linear(256 * 8 * 8, 512)
        self.act_fc2  = nn.Linear(512, 5)
        self.coord_conv1 = nn.Conv2d(256, 128, 3, padding=1)
        self.coord_conv2 = nn.Conv2d(128,  64, 3, padding=1)
        self.coord_conv3 = nn.Conv2d( 64,  32, 1)
        self.coord_conv4 = nn.Conv2d( 32,   1, 1)

    def forward(self, x):
        x = F.relu(self.conv1(x)); x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x)); feat = F.relu(self.conv4(x))
        a = self.act_pool(feat).flatten(1)
        action_logits = self.act_fc2(self.dropout(F.relu(self.act_fc1(a))))
        c = F.relu(self.coord_conv1(feat)); c = F.relu(self.coord_conv2(c))
        c = F.relu(self.coord_conv3(c))
        coord_logits = self.coord_conv4(c).flatten(1)
        return torch.cat([action_logits, coord_logits], dim=1)


# =============================================================================
# Frame encoding  (unchanged from v4)
# =============================================================================

def frame_to_tensor(frame: np.ndarray) -> torch.Tensor:
    if frame.shape != (64, 64):
        frame = frame[:64, :64] if frame.shape[0] >= 64 else np.pad(
            frame, ((0, 64-frame.shape[0]), (0, 64-frame.shape[1])), mode='edge')
    one_hot = torch.zeros(16, 64, 64, dtype=torch.float32)
    fc = np.clip(frame, 0, 15).astype(np.int64)
    one_hot.scatter_(0, torch.from_numpy(fc).unsqueeze(0), 1.0)
    cnt = np.bincount(fc.flatten(), minlength=16)
    bg  = int(cnt.argmax()); mx = max(cnt.max(), 1)
    bg_mask = (fc == bg).astype(np.float32)
    rarity  = np.zeros((64, 64), dtype=np.float32)
    for c in range(16):
        if cnt[c] > 0: rarity[fc == c] = 1.0 - cnt[c] / mx
    pad  = np.pad(fc, 1, mode='edge')
    edge = ((fc != pad[:-2,1:-1])|(fc != pad[2:,1:-1])|
            (fc != pad[1:-1,:-2])|(fc != pad[1:-1,2:])).astype(np.float32)
    rp = np.linspace(0,1,64,dtype=np.float32)[:,None].repeat(64,axis=1)
    cp = np.linspace(0,1,64,dtype=np.float32)[None,:].repeat(64,axis=0)
    aug = torch.from_numpy(np.stack([bg_mask,rarity,edge,rp,cp]))
    return torch.cat([one_hot, aug], dim=0)


# =============================================================================
# Reward shaping  (unchanged from v4)
# =============================================================================

def compute_reward(prev_frame, curr_frame, prev_hash, curr_hash, visited_hashes, bg):
    mask = np.ones((64,64),dtype=bool); mask[:2]=False; mask[62:]=False
    reward = 1.5 if curr_hash not in visited_hashes else -0.1
    if np.any((prev_frame != curr_frame) & mask): reward += 0.5
    def get_objs(f):
        o = {}
        for c in range(16):
            if c==bg: continue
            m=(f==c); n=int(m.sum())
            if 4<=n<=3000:
                ys,xs=np.where(m); o[c]=(float(xs.mean()),float(ys.mean()))
        return o
    po,co = get_objs(prev_frame), get_objs(curr_frame)
    moved = sum(1 for c,(cx,cy) in co.items()
                if c in po and 2<abs(cx-po[c][0])+abs(cy-po[c][1])<20)
    if moved: reward += 0.3 * min(moved, 3)
    return reward


# =============================================================================
# Agent
# =============================================================================

class MyAgent(Agent):
    """
    ARC3-Memory v5: object-centric TransitionMemory.

    Key change from v4:
      TransitionMemory now stores and queries using object-centric descriptors
      rather than flat color histograms. Two sub-buffers:
        - frame_mem: directional actions (where in frame → which action → reward)
        - click_mem: click actions (which object type → reward)
      The click_heatmap() overlaid on CNN coord logits is the main new signal:
      it places Gaussian bumps on objects that have historically been worth clicking,
      providing targeted click guidance without per-step LLM calls.
    """

    MAX_ACTIONS   = float('inf')
    _MAX_FRAMES   = 10
    WARMUP_STEPS  = 12
    BUFFER_SIZE   = 100_000
    BATCH_SIZE    = 64
    TRAIN_FREQ    = 10
    LR            = 3e-4
    EPS_START     = 0.20
    EPS_MIN       = 0.04
    EPS_DECAY     = 0.9998
    TOTAL_SECS    = 8 * 3600 - 300
    STUCK_THRESH  = 30
    MEM_FRAME_SIZE = 4000   # raised from 1000 — accumulates across game runs
    MEM_CLICK_SIZE = 4000   # raised from 1000 — accumulates across game runs
    MEM_K          = 8
    MEM_STRENGTH   = 0.8
    HEATMAP_SIGMA  = 4.0
    HEATMAP_STR    = 1.5

    # Class-level shared memory: game_type → TransitionMemory
    # Lives for the entire process lifetime, shared across all game instances
    # of the same game type. Keyed by the prefix before the first '-' in game_id.
    _SHARED_MEMORY: dict = {}

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        seed = int(time.time()*1e6) + hash(self.game_id) % 1_000_000
        random.seed(seed); np.random.seed(seed%(2**32-1)); torch.manual_seed(seed%(2**32-1))
        self.start_time = time.time()
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        # Derive game type: "ls20" from "ls20-abc123", "vc33" from "vc33-xyz"
        # Falls back to full game_id if no '-' present
        self.game_type = self.game_id.split('-')[0]

        print(f"[ARC3-v6] game={self.game_id} type={self.game_type} device={self.device}")

        self.GRID = 64
        self.net  = None
        self.opt  = None
        self.buffer = deque(maxlen=self.BUFFER_SIZE)

        # Attach to (or create) the shared memory for this game type
        if self.game_type not in MyAgent._SHARED_MEMORY:
            MyAgent._SHARED_MEMORY[self.game_type] = TransitionMemory(
                frame_maxsize=self.MEM_FRAME_SIZE,
                click_maxsize=self.MEM_CLICK_SIZE,
            )
            print(f"[ARC3-v6] Created new shared memory for game_type='{self.game_type}'")
        else:
            existing = MyAgent._SHARED_MEMORY[self.game_type]
            print(f"[ARC3-v6] Reusing shared memory for game_type='{self.game_type}' "
                  f"(frames={existing.frame_size} clicks={existing.click_size})")
        self.tmem = MyAgent._SHARED_MEMORY[self.game_type]

        # Per-step state
        self.prev_raw      = None
        self.prev_action   = None       # flat index or _UNDO_ACTION_IDX
        self.prev_hash     = ""
        self.prev_objects  = []         # objects extracted from prev frame
        self.visited_hashes = set()
        self.current_bg    = 0
        self.eps = self.EPS_START
        self.current_level      = -1
        self.level_action_count = 0
        self.unproductive        = 0
        self.checkpoint_hash     = None
        self.action_list = [
            GameAction.ACTION1, GameAction.ACTION2, GameAction.ACTION3,
            GameAction.ACTION4, GameAction.ACTION5,
        ]

    # ── Framework hooks ────────────────────────────────────────────────

    def append_frame(self, f: FrameData) -> None:
        self.frames.append(f)
        if len(self.frames) > self._MAX_FRAMES: self.frames = self.frames[-self._MAX_FRAMES:]
        if f.guid: self.guid = f.guid
        if hasattr(self, "recorder") and not self.is_playback:
            import json; self.recorder.record(json.loads(f.model_dump_json()))

    def is_done(self, frames, lf) -> bool:
        try:
            return lf.state is GameState.WIN or (time.time()-self.start_time) >= self.TOTAL_SECS
        except Exception: return True

    def choose_action(self, frames, lf: FrameData):
        try:
            return self._choose_action_impl(frames, lf)
        except Exception as e:
            traceback.print_exc()
            act = random.choice(self.action_list); act.reasoning = f"err:{e}"; return act

    # ── Helpers ────────────────────────────────────────────────────────

    def _level(self, f): return getattr(f,'score',None) or f.levels_completed
    def _get_raw(self, f): return np.array(f.frame, dtype=np.int64)[-1]
    def _hash(self, raw): return hashlib.md5(raw.tobytes()).hexdigest()[:16]

    def _reset_level(self) -> None:
        """Per-level reset. tmem is shared across levels AND game instances — never reset."""
        self.buffer.clear()
        self.visited_hashes.clear()
        self.prev_raw = self.prev_action = None
        self.prev_hash = ""; self.prev_objects = []
        self.level_action_count = 0; self.unproductive = 0
        self.checkpoint_hash = None; self.eps = self.EPS_START
        self.net = BaselineNet(in_channels=21, grid=self.GRID).to(self.device)
        self.opt = optim.Adam(self.net.parameters(), lr=self.LR)
        print(f"[ARC3-v6] Level {self.current_level} — "
              f"new CNN | shared tmem frames={self.tmem.frame_size} clicks={self.tmem.click_size}")

    def _avail_set(self, avail):
        return {int(a.value) if hasattr(a,'value') else int(a) for a in avail}

    def _safe_fallback(self, av):
        choices = [a-1 for a in av if 1<=a<=5]
        if choices: return random.choice(choices), None
        if 6 in av: return 5, (random.randint(0,63), random.randint(0,63))
        return 0, None

    # ── Main loop ──────────────────────────────────────────────────────

    def _choose_action_impl(self, frames, lf: FrameData):
        lvl = self._level(lf)
        if lvl != self.current_level:
            self.current_level = lvl; 
            self._reset_level()

        if lf.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
            self.prev_raw = self.prev_action = None; 
            self.prev_objects = []
            act = GameAction.RESET; 
            act.reasoning = "reset"; 
            return act

        raw       = self._get_raw(lf)
        curr_hash = self._hash(raw)
        avail     = getattr(lf, 'available_actions', None) or []
        av        = self._avail_set(avail)
        cnt       = np.bincount(raw.flatten(), minlength=16)
        self.current_bg = int(cnt.argmax())

        # Extract objects for current frame
        cur_objects = extract_objects(raw, self.current_bg)

        undo_avail = 7 in av

        # ── Experience + memory store ──────────────────────────────────
        if self.prev_raw is not None and self.prev_action is not None:
            frame_changed = not np.array_equal(self.prev_raw[2:62], raw[2:62])
            reward = compute_reward(
                self.prev_raw, raw, self.prev_hash, curr_hash,
                self.visited_hashes, self.current_bg)

            self.buffer.append({'s': self.prev_raw.copy(), 'a': self.prev_action, 'r': reward})

            # Store in memory ONLY when frame changed and not UNDO 
            if frame_changed and self.prev_action != _UNDO_ACTION_IDX:
                prev_is_click = self.prev_action >= 5
                if prev_is_click:
                    # Decode click position from flat index
                    ci = self.prev_action - 5
                    cy, cx = ci // self.GRID, ci % self.GRID
                    clicked_obj = find_object_at(self.prev_objects, cy, cx)
                    if clicked_obj is not None:
                        self.tmem.store_click(object_descriptor(clicked_obj), reward)
                else:
                    # Directional action — store frame-level descriptor
                    prev_fdesc = frame_descriptor_v2(self.prev_objects, self.current_bg)
                    self.tmem.store_directional(prev_fdesc, self.prev_action, reward)

            if frame_changed: self.unproductive=0; self.checkpoint_hash=curr_hash
            else: self.unproductive += 1

        self.visited_hashes.add(curr_hash)

        # ── Undo if stuck ──────────────────────────────────────────────
        if undo_avail and self.unproductive >= self.STUCK_THRESH:
            self.unproductive = 0
            act = GameAction.ACTION7; act.reasoning = "undo_stuck"
            self._update_prev(raw, curr_hash, _UNDO_ACTION_IDX, cur_objects)
            return act

        # ── Action selection ───────────────────────────────────────────
        # FIX B: memory bias ONLY in CNN policy phase
        if self.level_action_count < self.WARMUP_STEPS:
            action_idx, coords = self._heuristic(raw, av, self.level_action_count, cur_objects)
        elif random.random() < self.eps:
            action_idx, coords = self._random_action(av)          # FIX B: pure uniform
            self.eps = max(self.EPS_MIN, self.eps * self.EPS_DECAY)
        else:
            action_idx, coords = self._cnn_action(raw, av, cur_objects)  # memory bias here
            self.eps = max(self.EPS_MIN, self.eps * self.EPS_DECAY)

        if self.level_action_count % self.TRAIN_FREQ == 0: self._train()
        self.level_action_count += 1

        # ── Build GameAction ───────────────────────────────────────────
        if action_idx < 5:
            sel = self.action_list[action_idx]; flat_idx = action_idx
            sel.reasoning = f"act{action_idx+1} eps={self.eps:.3f}"
        else:
            y, x = coords; sel = GameAction.ACTION6
            sel.set_data({'x': int(x), 'y': int(y)})
            flat_idx = 5 + int(y)*self.GRID + int(x)
            sel.reasoning = f"click({x},{y})"

        self._update_prev(raw, curr_hash, flat_idx, cur_objects)
        return sel

    # ── Action selection helpers ───────────────────────────────────────

    def _heuristic(self, frame, av, step, objects):
        """FIX C: fixed sequential warmup. No memory bias. Object list for click targeting."""
        if step < 4:
            for d in range(1, 5):
                if d in av: return d-1, None

        if 6 in av and objects:
            # Click non-background objects in size order (smallest first = most salient)
            targets = [(obj['cx'], obj['cy'], obj['size']) for obj in objects]
            targets.sort(key=lambda t: t[2])
            pidx = step - 4
            if 0 <= pidx < len(targets):
                tx, ty, _ = targets[pidx]
                return 5, (int(ty), int(tx))

        return self._safe_fallback(av)

    def _random_action(self, av):
        """FIX B: pure uniform random."""
        has6 = 6 in av
        disc = [a-1 for a in av if 1<=a<=5]
        if not disc and not has6: return self._safe_fallback(av)
        if has6 and (not disc or random.random() < 0.4):
            return 5, (random.randint(0,63), random.randint(0,63))
        if disc: return random.choice(disc), None
        return self._safe_fallback(av)

    def _cnn_action(self, frame, av, objects):
        """CNN policy with object-centric memory bias injected into logits."""
        tensor = frame_to_tensor(frame).unsqueeze(0).to(self.device)
        with torch.no_grad():
            logits = self.net(tensor).squeeze(0).clone()   # (4101,)

        # Discrete action bias from frame memory
        fdesc = frame_descriptor_v2(objects, self.current_bg)
        d_bias = self.tmem.discrete_bias(fdesc, k=self.MEM_K, strength=self.MEM_STRENGTH)
        logits[:5] += torch.from_numpy(d_bias).to(self.device)

        # Click heatmap from click object memory — add to coord logits
        heatmap = self.tmem.click_heatmap(
            objects, k=self.MEM_K, sigma=self.HEATMAP_SIGMA, strength=self.HEATMAP_STR
        )
        logits[5:] += torch.from_numpy(heatmap.flatten()).to(self.device)

        return self._sample_from_logits(logits, av)

    def _sample_from_logits(self, logits, av):
        """FIX 3 (v3): fallback constrained to available action slots."""
        act_logits   = logits[:5].clone()
        coord_logits = logits[5:].clone()
        has6 = False
        mask = torch.full((5,), float('-inf'), device=logits.device)
        for a in av:
            aid = int(a.value) if hasattr(a,'value') else int(a)
            if 1<=aid<=5: mask[aid-1] = 0.0
            elif aid==6:  has6 = True
        act_logits = act_logits + mask
        if not has6: coord_logits = coord_logits + float('-inf')

        act_probs   = torch.sigmoid(act_logits)
        coord_probs = torch.sigmoid(coord_logits) / (self.GRID**2)
        all_probs   = torch.cat([act_probs, coord_probs])
        s = all_probs.sum().item()

        if s < 1e-8:
            fallback = torch.zeros_like(all_probs)
            for a in av:
                aid = int(a.value) if hasattr(a,'value') else int(a)
                if 1<=aid<=5: fallback[aid-1] = 1.0
                elif aid==6:  fallback[5:] = 1.0/4096
            s2 = fallback.sum().item()
            all_probs = fallback/s2 if s2 > 0 else torch.ones_like(all_probs)/len(all_probs)

        p = (all_probs.double() / all_probs.double().sum()).cpu().numpy()
        idx = np.random.choice(len(p), p=p)
        if idx < 5: return idx, None
        ci = idx-5; return 5, (ci//self.GRID, ci%self.GRID)

    # ── CNN training ───────────────────────────────────────────────────

    def _train(self) -> None:
        if self.net is None or len(self.buffer) < self.BATCH_SIZE: return
        idx   = np.random.choice(len(self.buffer), self.BATCH_SIZE, replace=False)
        batch = [self.buffer[i] for i in idx]
        states  = torch.stack([frame_to_tensor(e['s']).to(self.device) for e in batch])
        actions = torch.tensor([e['a'] for e in batch], dtype=torch.long, device=self.device)
        rewards = torch.tensor([e['r'] for e in batch], dtype=torch.float32, device=self.device)
        targets = torch.sigmoid(rewards)
        self.opt.zero_grad()
        logits  = self.net(states)
        a_clamp = actions.clamp(0, logits.size(1)-1)
        sel     = logits.gather(1, a_clamp.unsqueeze(1)).squeeze(1)
        loss    = F.binary_cross_entropy_with_logits(sel, targets)
        probs   = torch.sigmoid(logits)
        loss    = loss - 1e-4*probs[:,:5].mean() - 1e-5*probs[:,5:].mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.net.parameters(), 1.0)
        self.opt.step()

    # ── Bookkeeping ────────────────────────────────────────────────────

    def _update_prev(self, raw, curr_hash, action_idx, objects) -> None:
        self.prev_raw    = raw.copy()
        self.prev_hash   = curr_hash
        self.prev_action = action_idx
        self.prev_objects = objects   # needed to find clicked object next step



In [ ]:
import os
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    !curl --fail --retry 999 --retry-all-errors \
          --retry-delay 5 --retry-max-time 600 \
          http://gateway:8001/api/games
    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents \
           /kaggle/working/ARC-AGI-3-Agents
    !cp /kaggle/working/my_agent.py \
        /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py','w') as f:
        f.write('from typing import Type\nfrom dotenv import load_dotenv\n'
                'from .agent import Agent, Playback\nfrom .swarm import Swarm\n'
                'from .templates.random_agent import Random\n'
                'from .templates.my_agent import MyAgent\nload_dotenv()\n'
                'AVAILABLE_AGENTS: dict[str, Type[Agent]] = {"random": Random, "myagent": MyAgent}\n')
    with open('/kaggle/working/ARC-AGI-3-Agents/.env','w') as f:
        f.write('SCHEME=http\nHOST=gateway\nPORT=8001\n'
                'ARC_API_KEY=test-key-123\nARC_BASE_URL=http://gateway:8001/\n'
                'OPERATION_MODE=online\nRECORDINGS_DIR=/kaggle/working/server_recording\n')
    !cd /kaggle/working/ARC-AGI-3-Agents && MPLBACKEND=agg python main.py --agent myagent

In [ ]:
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import pandas as pd
    pd.DataFrame(data=[['1_0','1',True,1]],
                 columns=['row_id','game_id','end_of_game','score']).to_parquet('/kaggle/working/submission.parquet', index=False)
    print('Non-competition mode: dummy submission.parquet written.')